# 大学生 AI 应用能力差异分析：PCA + TOPSIS + 马尔可夫链

本 notebook 是《马太效应视域下大学生应用 AI 的差异与对策研究》的可复现分析流程，基于 600 份有效问卷的处理后数据。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

data = pd.read_csv('../data/processed_data.csv')
print('数据维度:', data.shape)
data.head()

## 1. 指标分组（五维度评价体系）

将 24 个题项归入五个维度，与论文口径一致。

In [ ]:
DIMENSIONS = {
    '基础使用与场景分布': [
        '使用AI工具的频率',
        '最常使用AI的领域_课程作业', '最常使用AI的领域_学术研究',
        '最常使用AI的领域_技能学习', '最常使用AI的领域_娱乐创作',
    ],
    '工具掌握与交互深度': [
        '使用AI工具的主要方式_手机APP', '使用AI工具的主要方式_电脑客户端',
        '使用AI工具的主要方式_网页端', '解决单一复杂问题时的对话交互次数',
        '当AI首次提供的答案不符合预期时的主要操作_追加提问细化需求',
        '当AI首次提供的答案不符合预期时的主要操作_主动提供更多背景信息',
        '当AI首次提供的答案不符合预期时的主要操作_切换提问方式重新尝试',
    ],
    '深度应用能力': [
        '使用AI技术完成复杂任务情况_编程项目开发',
        '使用AI技术完成复杂任务情况_创意作品生成',
        '使用AI技术完成复杂任务情况_仅用于简单问答或娱乐',
        '使用AI技术完成复杂任务情况_课程论文/实验报告撰写',
        '使用AI技术完成复杂任务情况_学术研究数据分析',
        '因“AI试错成本”放弃高阶应用学习情况',
    ],
    '资源获取渠道与限制': [
        '学习AI技术的主要渠道_同学/朋友交流', '学习AI技术的主要渠道_从未系统学习',
        '学习AI技术的主要渠道_学校课程/讲座', '校内AI教育资源获取难度',
    ],
    '社交与实践机会': [
        'AI行业实习或项目参与情况', 'AI领域的学术社群或会议参与情况',
    ],
}
ITEMS = [c for cols in DIMENSIONS.values() for c in cols]
print('指标总数:', len(ITEMS))

## 2. 数据编码

对 6 个有序题项进行数值编码，多选哑变量已为 0/1。

In [ ]:
ORDINAL_MAPPING = {
    '使用AI工具的频率': {'每天多次': 1, '每周几次': 2, '每月几次': 3, '几乎不用': 4},
    '解决单一复杂问题时的对话交互次数': {'中途放弃': 0, '3轮以内快速解决': 1, '4-6轮常规交流': 2, '7轮以上深度探讨': 3},
    '因“AI试错成本”放弃高阶应用学习情况': {'是': 1, '否': 0},
    '校内AI教育资源获取难度': {'非常容易': 1, '容易': 2, '一般': 3, '困难': 4, '非常困难': 5},
    'AI行业实习或项目参与情况': {'是': 1, '否': 0},
    'AI领域的学术社群或会议参与情况': {'是': 1, '否': 0},
}

# 成本型指标：数值越大代表能力越弱，TOPSIS 中作为负向指标处理
COST_ITEMS = [
    '使用AI工具的频率',
    '因“AI试错成本”放弃高阶应用学习情况',
    '校内AI教育资源获取难度',
    '学习AI技术的主要渠道_从未系统学习',
    '使用AI技术完成复杂任务情况_仅用于简单问答或娱乐',
]

encoded = data.copy()
for col, mapping in ORDINAL_MAPPING.items():
    encoded[col] = encoded[col].map(mapping).astype(float)

X = encoded[ITEMS].astype(float).copy()
print('编码后缺失值总数:', int(X.isnull().sum().sum()))
X = X.fillna(X.median())
X.shape

## 3. PCA 方差加权权重

对标准化后的指标矩阵做特征值分解，用「载荷平方 × 方差贡献率」计算客观权重。

In [ ]:
scaler = StandardScaler()
Z = scaler.fit_transform(X)
cov = np.cov(Z, rowvar=False)
eigenvalues, eigenvectors = np.linalg.eig(cov)

order = eigenvalues.argsort()[::-1]
eigenvalues = eigenvalues[order].real
eigenvectors = eigenvectors[:, order].real
explained = eigenvalues / eigenvalues.sum()
cumulative = np.cumsum(explained)
n_components = int(np.argmax(cumulative >= 0.99) + 1)
n_components = min(len(ITEMS), max(1, n_components))

loadings = eigenvectors * np.sqrt(eigenvalues)
loadings_df = pd.DataFrame(loadings[:, :n_components],
                           columns=[f'PC{i+1}' for i in range(n_components)], index=ITEMS)

weights = pd.Series(0.0, index=ITEMS)
for item in ITEMS:
    w = 0.0
    for i in range(n_components):
        w += (loadings_df.loc[item, f'PC{i+1}'] ** 2) * explained[i]
    weights[item] = w
weights = weights / weights.sum()

weight_df = weights.rename('权重').sort_values(ascending=False)
print('保留主成分数:', n_components)
print(weight_df.head(12))

plt.figure(figsize=(10, 6))
weight_df.head(20)[::-1].plot(kind='barh')
plt.title('指标客观权重（PCA 方差加权，Top 20）')
plt.tight_layout()
plt.savefig('../figures/item_weights.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. TOPSIS 综合评分

用标准化矩阵 × 权重构造加权决策矩阵，按正负理想解计算贴近度，作为 AI 应用能力得分。

In [ ]:
W = weights.loc[ITEMS].values
V = Z * W  # 加权标准化决策矩阵

ideal_best = np.where(np.isin(ITEMS, COST_ITEMS), V.min(axis=0), V.max(axis=0))
ideal_worst = np.where(np.isin(ITEMS, COST_ITEMS), V.max(axis=0), V.min(axis=0))

d_best = np.sqrt(((V - ideal_best) ** 2).sum(axis=1))
d_worst = np.sqrt(((V - ideal_worst) ** 2).sum(axis=1))
score = d_worst / (d_best + d_worst)

encoded['score'] = score
print('得分区间:', round(score.min(), 3), '~', round(score.max(), 3))

# 各维度得分（维度内指标标准化后加权求和，便于观察短板）
dim_scores = {}
for dim, cols in DIMENSIONS.items():
    sub = X[cols].apply(lambda s: (s - s.mean()) / (s.std() + 1e-9))
    dim_scores[dim] = sub.mean(axis=1).mean()
print('\n各维度平均标准化得分:')
for k, v in sorted(dim_scores.items(), key=lambda x: x[1]):
    print(f'  {k}: {v:.3f}')

## 5. 等频分箱（低 / 中 / 高三档）

按论文做法将综合得分等频分为三档，用于马尔可夫转移分析。

In [ ]:
encoded['Level'] = pd.qcut(encoded['score'], q=3, labels=['低', '中', '高'], duplicates='drop')
print(encoded['Level'].value_counts().sort_index())

## 6. 群体差异可视化

按性别、学历、学校层次、家庭年收入等分组观察得分分布。

In [ ]:
group_cols = ['性别', '学历', '学校层次', '家庭年收入']
fig, axes = plt.subplots(1, len(group_cols), figsize=(16, 4), sharey=True)
for ax, col in zip(axes, group_cols):
    order = encoded.groupby(col)['score'].median().sort_values().index
    sns.boxplot(data=encoded, x=col, y='score', order=order, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('../figures/group_scores_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. 马尔可夫链状态转移

把相邻样本的能力层级变化视作状态转移，估计不同群体在「低 / 中 / 高」三档间的转移概率。

In [ ]:
def build_level_markov_matrix(level_sequence, levels):
    matrix = pd.DataFrame(0.0, index=levels, columns=levels)
    seq = list(level_sequence)
    for cur, nxt in zip(seq[:-1], seq[1:]):
        matrix.loc[cur, nxt] += 1
    row_sum = matrix.sum(axis=1).replace(0, 1)
    return (matrix.div(row_sum, axis=0), levels)

levels = ['低', '中', '高']

def plot_transition(feature):
    states = sorted(encoded[feature].dropna().unique())
    n = len(states)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, state in zip(axes, states):
        sub = encoded[encoded[feature] == state].sort_values('score')
        if len(sub) < 2:
            ax.axis('off'); continue
        mat, _ = build_level_markov_matrix(sub['Level'], levels)
        sns.heatmap(mat, annot=True, fmt='.2f', cmap='Blues', ax=ax,
                    vmin=0, vmax=1, cbar=(state == states[-1]))
        ax.set_title(f'{feature} = {state}')
    fig.suptitle(f'{feature} 的马太效应转移矩阵', fontsize=14)
    plt.tight_layout()
    plt.show()

plot_transition('学历')
plot_transition('性别')

## 结论摘要

- 大学生 AI 应用能力在「社交与实践机会」维度得分最低，是普遍短板。
- 经管类、家庭年收入 40 万以上群体的能力分层与「强者愈强」特征更明显。
- 本科生的能力向上流动概率相对更高；资源与动机的匹配程度决定流动方向。